# Udaplay Project
## Part 02 - Agent

Now we build **UdaPlay**, an agent that answers game questions.

For every question it goes through the same simple steps:

```
retrieve_game  ->  evaluate_retrieval  ->  (game_web_search, only if needed)  ->  final answer
```

1. **retrieve_game**: search our own vector database from Part 1.
2. **evaluate_retrieval**: a second LLM acts as a judge and decides whether those results answer the question.
3. **game_web_search**: only if the judge says *not useful*, search the web with Tavily.
4. **Answer**: a structured answer with a **source** and a **confidence** level.

The agent runs on the course's `StateMachine` (`lib/agents.py`), and it keeps conversation history per session. That's why follow-up questions work.

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import json
from typing import Literal

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, AIMessage
from lib.tooling import tool

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("All 3 environment variables are set.")

### Tools
Three tools. Each prints what it is doing, so you can follow the agent's steps in the output.

#### Retrieve Game Tool
Searches the vector DB and returns the 3 closest games.

In [ ]:
# Load the database built in Part 1 (same path, same name, same embedding function)
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-ada-002",
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
print("Games in vector DB:", collection.count())


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    print(f"\n[STEP 1 - retrieve_game] query = {query!r}")
    results = collection.query(query_texts=[query], n_results=3)
    docs = []
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        docs.append({**meta, "distance": round(dist, 3)})
        print(f"   found: {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']})  distance={dist:.3f}")
    return docs

#### Evaluate Retrieval Tool
An "LLM as judge": it reads the question and the retrieved games and returns a structured `EvaluationReport` (useful yes/no, confidence, reason).

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="True if the documents are enough to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description="How sure the judge is")
    description: str = Field(description="Short explanation of the decision")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high / medium / low
    - description: description about the evaluation result
    """
    print(f"\n[STEP 2 - evaluate_retrieval] question = {question!r}")
    judge = LLM(model="gpt-4o-mini", temperature=0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Only say useful=true if the documents directly contain the answer. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\nDocuments:\n" + "\n".join(f"- {d}" for d in retrieved_docs)
    )
    response = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(response.content)
    print(f"   useful={report.useful}  confidence={report.confidence}")
    print(f"   reason: {report.description}")
    return report.model_dump()

#### Game Web Search Tool
The fallback. It uses Tavily to search the web and returns the page URLs so the answer can cite them.

In [ ]:
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    print(f"\n[STEP 3 - game_web_search] question = {question!r}")
    response = tavily.search(
        query=question,
        search_depth="advanced",
        max_results=5,
        include_answer=True,
    )
    results = [
        {"title": r["title"], "url": r["url"], "content": r["content"][:500]}
        for r in response.get("results", [])
    ]
    for r in results:
        print(f"   source: {r['url']}")
    return {"answer": response.get("answer"), "results": results}

### Agent
The `Agent` class from `lib/agents.py` is a **state machine**:

```
entry -> message_prep -> llm_processor --(tool calls?)--> tool_executor -> llm_processor ... -> termination
```

The LLM picks tools according to the instructions below. Every question in the same `session_id` adds to one shared message history (short-term memory).

In [ ]:
INSTRUCTIONS = """
You are UdaPlay, a research assistant for video games.

Follow these steps for EVERY question, in this order:
1. Call retrieve_game to search the internal game database.
   If the question refers to something earlier in the conversation (like "it" or "that game"),
   rewrite it into a full question first, using the conversation history.
2. Call evaluate_retrieval with the question and the retrieved documents.
3. ONLY if evaluate_retrieval says useful = false, call game_web_search.
   If useful = true, do NOT search the web.
4. Write the final answer in exactly this format:

**Answer:** <a clear, short answer in natural language>
**Source:** <"Internal database: <game name(s)>" or "Web: <URL(s)>" - always include URLs for web answers>
**Confidence:** <high / medium / low, with one short reason>
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0,
)
print(agent.workflow)

In [ ]:
def ask(question, session_id="demo"):
    """Ask the agent a question, then print a short report of what it did."""
    print("=" * 90)
    print("QUESTION:", question)
    print("=" * 90)
    run = agent.invoke(question, session_id=session_id)
    messages = run.get_final_state()["messages"]

    # Only look at messages from THIS question (after the last user message)
    last_user = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage))
    tools_used = [
        call.function.name
        for m in messages[last_user:]
        if isinstance(m, AIMessage) and m.tool_calls
        for call in m.tool_calls
    ]
    print("\n--- REPORT ---")
    print("Tools used, in order:", " -> ".join(tools_used) or "none")
    print("Web search needed:", "game_web_search" in tools_used)
    print()
    print(messages[-1].content)
    print()
    return run

### Run the agent
**Query 1 (in the local dataset).** We expect: retrieve -> evaluate (useful) -> answer. No web search.

In [ ]:
ask("When was Pokémon Gold and Silver released?");

**Query 2 (in the local dataset).** Super Mario 64 is in our data.

In [ ]:
ask("Which one was the first 3D platformer Mario game?");

**Query 3 (NOT in the local dataset).** Mortal Kombat X isn't one of our 15 games, so the judge should say *not useful* and the agent should search the web and cite URLs.

In [ ]:
ask("Was Mortal Kombat X released for PlayStation 5?");

**Query 4 (local).** Publisher question from the project scenario.

In [ ]:
ask("Who published Grand Theft Auto: San Andreas?");

### Conversation memory
This follow-up just says *"it"*. The agent only understands it because it remembers the previous question in the same session (`demo`).

In [ ]:
ask("What platform was it released on, and in which year?");

**Query 6 (needs the web).** Current news, which a static dataset can't know.

In [ ]:
ask("What is Rockstar Games working on right now?");

A **new session** has no history, so the agent can't know what "it" means. This shows that memory is kept per session.

In [ ]:
ask("What platform was it released on, and in which year?", session_id="new_session");

### Summary
* Questions about the 15 local games were answered from the **vector database**, with no web search.
* Questions outside the dataset (Mortal Kombat X, Rockstar's current work) were judged **not useful** and fell back to **Tavily web search**, citing URLs.
* Every answer has a **source** and a **confidence** level.
* The agent remembers earlier questions **within a session**.